## Dependencies

In [1]:
## libraries
import os
import sys
import pandas as pd
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.evaluators.config import FEAT_X, FEAT_Z, TARGET
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.decomposing import eval_separability
from src.evaluators.decomposing import eval_exhaustiveness

## Initialization

In [3]:
## setup
data = load_processed_data()
models = load_estimators()

## Decomposition Tests

In [4]:
## additive vs interaction vs joint specification under logo-cv
separability, predictions = eval_separability(
    data = data,
    models = models,
    feat_x = FEAT_X,
    feat_z = FEAT_Z,
    target = TARGET,
    n_jobs = -1,
)

  linear_convex: additive...   linear_quantile: additive...   linear_laws: additive...   xgboost_quantile: additive... interaction... interaction...   forest_quantile: additive... interaction...   boosted_quantile: additive... joint... joint... interaction_joint... interaction_joint... capacity_only... capacity_only... dynamics_only... dynamics_only... done.
done.
joint... interaction_joint... capacity_only... dynamics_only... done.
interaction... interaction...   neural_quantile: additive...   neural_convex: additive...   neural_expectile: additive... interaction... joint... interaction_joint... joint... capacity_only... dynamics_only... interaction_joint... done.
capacity_only... dynamics_only... joint... interaction_joint... done.
capacity_only... dynamics_only... done.
interaction... interaction... interaction... joint... joint... joint... interaction_joint... interaction_joint... interaction_joint... capacity_only... capacity_only... capacity_only... dynamics_only... dynamics_only

In [11]:
## display settings
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
 
## specification order
spec_order = ["additive", "interaction", "interaction_joint", "joint",
              "capacity_only", "dynamics_only"]

# ## --- by domain (averaged across models) ---
# domain_agg = separability.groupby(["group", "specification"])["ei"].mean().unstack()
# domain_agg = domain_agg[[c for c in spec_order if c in domain_agg.columns]]
# print("=== EI by Domain ===")
# display(domain_agg.round(4))

# ## --- by model (averaged across domains) ---
# model_agg = separability.groupby(["model", "specification"])["ei"].mean().unstack()
# model_agg = model_agg[[c for c in spec_order if c in model_agg.columns]]
# print("\n=== EI by Model ===")
# display(model_agg.round(4))

## --- overall summary ---
summary = separability.groupby("specification")[["ei", "vr", "mv", "ms", "ea"]].mean()
summary = summary.reindex([s for s in spec_order if s in summary.index])
print("\n=== Mean Frontier Metrics by Specification ===")
display(summary.round(4))


=== Mean Frontier Metrics by Specification ===


,ei,vr,mv,ms,ea
specification,,,,,
additive,0.718000,0.093300,3.900300,5.411500,0.922100
interaction,0.713200,0.062200,0.460900,1377.082500,140.081500
interaction_joint,0.713000,0.120000,0.548300,5.205300,0.978400
joint,0.723000,0.097800,0.624000,5.161600,0.962500
capacity_only,0.704800,0.111100,3.976000,5.397400,0.990000
dynamics_only,0.724700,0.124400,0.515200,4.686600,0.876200


### Exhaustiveness (Residual Irrelevance of X for Slack)

In [5]:
## exhaustiveness: does C(X) fully absorb X's contribution?
exhaustiveness, exhaust_predictions = eval_exhaustiveness(
    data = data,
    models = models,
    feat_x = FEAT_X,
    feat_z = FEAT_Z,
    target = TARGET,
    n_jobs = min(4, os.cpu_count() or 1),
)

  linear_laws: C(X)...   linear_convex:   linear_quantile: C(X)... C(X)...   forest_quantile: C(X)... X_to_slack... X_to_slack... Z_to_slack... Z_to_slack... done.
done.
  xgboost_quantile: C(X)...   boosted_quantile: C(X)... X_to_slack... X_to_slack... X_to_slack... Z_to_slack... X_to_slack... Z_to_slack... Z_to_slack... done.
done.
done.
Z_to_slack... done.
  neural_quantile: C(X)...   neural_expectile: C(X)...   neural_convex: C(X)... X_to_slack... X_to_slack... X_to_slack... Z_to_slack... Z_to_slack... Z_to_slack... done.
done.
done.


In [7]:
from scipy.stats import wilcoxon

## --- aggregate to domain level: mean abs_error per (model, residual_features, group) ---
domain_err = exhaust_predictions.groupby(
    ["model", "residual_features", "group"]
)["abs_error"].mean().reset_index()

x_slack_err = domain_err.query(
    "residual_features == 'X_to_slack'"
).set_index(["model", "group"])["abs_error"]

z_slack_err = domain_err.query(
    "residual_features == 'Z_to_slack'"
).set_index(["model", "group"])["abs_error"]

delta = (x_slack_err - z_slack_err).dropna()
n = len(delta)
n_pos = int((delta > 0).sum())
n_neg = int((delta < 0).sum())
stat, p = wilcoxon(delta.values, alternative = "greater")
r_effect = 1 - (2 * stat) / (n * (n + 1) / 2)

if p < 0.01:
    sig = "**"
elif p < 0.05:
    sig = "*"
elif p < 0.10:
    sig = "†"
else:
    sig = ""

exhaust_result = pd.DataFrame([{
    "Hypothesis": "Z → slack has lower error than X → slack",
    "n": n,
    "Z Better": n_pos,
    "X Better": n_neg,
    "Mean Δ |error|": delta.mean(),
    "Median Δ |error|": delta.median(),
    "Effect Size (r)": r_effect,
    "p (one-sided)": p,
    "Sig.": sig,
}])

print(f"\n=== Exhaustiveness: Residual Predictive Power of X for Slack (n = {n}) ===")
print("H₁: After extracting C(X), topology (X) has negligible residual predictive power for the slack compared to dynamics (Z)")
print("C(X) absorbs topology's contribution to y*; the remaining slack is better explained by dynamics\n")
display(
    exhaust_result.style
        .format({
            "Mean Δ |error|": "{:+.4f}",
            "Median Δ |error|": "{:+.4f}",
            "Effect Size (r)": "{:+.3f}",
            "p (one-sided)": "{:.4f}",
        })
        .set_caption("** p < 0.01, * p < 0.05, † p < 0.10")
        .hide(axis = "index")
)

## --- summary by feature set (domain-level, n pairs each) ---
print(f"\n=== Mean |error|: X → slack vs Z → slack (domain-level) ===")
summary = domain_err.groupby(
    "residual_features"
)["abs_error"].agg(["mean", "std", "count"])
display(summary.round(4))


=== Exhaustiveness: Residual Predictive Power of X for Slack (n = 45) ===
H₁: After extracting C(X), topology (X) has negligible residual predictive power for the slack compared to dynamics (Z)
C(X) absorbs topology's contribution to y*; the remaining slack is better explained by dynamics



Hypothesis,n,Z Better,X Better,Mean Δ |error|,Median Δ |error|,Effect Size (r),p (one-sided),Sig.
Z → slack has lower error than X → slack,45,28,17,+268.2459,+0.6554,-0.459,0.0033,**



=== Mean |error|: X → slack vs Z → slack (domain-level) ===


,mean,std,count
residual_features,,,
X_to_slack,278.267700,1633.793300,45
Z_to_slack,10.021800,17.962500,45


### Sufficiency (Non-Inferiority Ceiling Tests)

In [8]:
import numpy as np
from scipy.stats import wilcoxon, ttest_1samp, binomtest

## --- pairwise ceiling tests: additive vs each relaxed spec ---
ceiling_specs = ["interaction", "interaction_joint", "joint",
                 "capacity_only", "dynamics_only"]
delta = 0.05  ## non-inferiority margin (EI points)

additive_ei = separability.query(
    "specification == 'additive'"
).set_index(["model", "group"])["ei"]

gap_by_spec = []
rows = []
for spec in ceiling_specs:
    spec_ei = separability.query(
        "specification == @spec"
    ).set_index(["model", "group"])["ei"]

    ## directed gap: positive = relaxed spec is better
    gap = (spec_ei - additive_ei).dropna()
    gap_by_spec.append(gap.rename(spec))
    n = len(gap)
    n_spec_better = int((gap > 0).sum())
    n_additive_better = int((gap <= 0).sum())

    ## canonical non-inferiority tests: H0: gap >= delta vs H1: gap < delta
    ## 1. paired t-test against margin
    t_stat, p_t = ttest_1samp(gap.values, popmean = delta, alternative = "less")
    
    ## 2. wilcoxon signed-rank test against margin
    w_stat, p_w = wilcoxon(gap.values - delta, alternative = "less")

    ## 3. sign test against margin (assumption-free)
    n_above = int((gap.values >= delta).sum())
    p_sign = binomtest(n_above, n, p = 0.5, alternative = "less").pvalue

    ## using wilcoxon as the primary p-value for reporting pairwise (robust to non-normality)
    p_ni = p_w 

    if p_ni < 0.01:
        sig = "**"
    elif p_ni < 0.05:
        sig = "*"
    elif p_ni < 0.10:
        sig = "†"
    else:
        sig = ""

    rows.append({
        "Specification": spec,
        "n": n,
        "Spec Better": n_spec_better,
        "Additive Better": n_additive_better,
        "Mean Δ EI": gap.mean(),
        "Median Δ EI": gap.median(),
        "t-test p": p_t,
        "Wilcoxon p": p_w,
        "Sign p": p_sign,
        "Sig.": sig,
    })

suff_result = pd.DataFrame(rows)

print(f"=== Sufficiency: Pairwise Non-Inferiority Ceiling Tests (n = {n}, Δ = {delta}) ===")
print(f"H₁: Relaxed specifications do not improve over additive C(X) + R(Z) by more than {delta} EI")
print("Each row tests one relaxed alternative against additive\n")
display(
    suff_result.style
        .format({
            "Mean Δ EI": "{:+.4f}",
            "Median Δ EI": "{:+.4f}",
            "t-test p": "{:.4f}",
            "Wilcoxon p": "{:.4f}",
            "Sign p": "{:.4f}",
        })
        .set_caption("** p < 0.01, * p < 0.05, † p < 0.10 (based on Wilcoxon)")
        .hide(axis = "index")
)

=== Sufficiency: Pairwise Non-Inferiority Ceiling Tests (n = 45, Δ = 0.05) ===
H₁: Relaxed specifications do not improve over additive C(X) + R(Z) by more than 0.05 EI
Each row tests one relaxed alternative against additive



Specification,n,Spec Better,Additive Better,Mean Δ EI,Median Δ EI,t-test p,Wilcoxon p,Sign p,Sig.
interaction,45,23,22,-0.0048,+0.0023,0.0246,0.0012,0.0001,**
interaction_joint,45,15,30,-0.0049,-0.0094,0.0254,0.0040,0.0001,**
joint,45,19,26,+0.0051,-0.0084,0.0583,0.0086,0.0004,**
capacity_only,45,16,29,-0.0131,-0.0102,0.0026,0.0004,0.0001,**
dynamics_only,45,20,25,+0.0067,-0.0061,0.0614,0.0060,0.0001,**


In [9]:
## --- absolute frontier quality of additive model ---
additive_summary = separability.query(
    "specification == 'additive'"
).groupby("model")[["ei", "vr", "mv", "ms", "ea"]].mean()

ei_mean = additive_summary["ei"].mean()
ei_std = additive_summary["ei"].std()
n_sig = (suff_result["Wilcoxon p"] < 0.05).sum()
max_pairwise_p = suff_result["Wilcoxon p"].max()

print(f"\nMean EI across models: {ei_mean:.4f} ± {ei_std:.4f}")
print(f"Max pairwise ceiling gap: {suff_result['Mean Δ EI'].max():+.4f}")
print(f"Non-inferiority margin: Δ = {delta}")
print(
    f"Additive is non-inferior to {n_sig}/{len(suff_result)} alternative specifications (p < 0.05)"
)
print(f"Worst pairwise p-value: {max_pairwise_p:.4f}")


Mean EI across models: 0.7180 ± 0.0811
Max pairwise ceiling gap: +0.0067
Non-inferiority margin: Δ = 0.05
Additive is non-inferior to 5/5 alternative specifications (p < 0.05)
Worst pairwise p-value: 0.0086
